# Osarumen Okhiku — Courts and Rinks Sub-metric
### CS 1656 Final Project | Group 4 — Fun Finder Pittsburgh

**Research Question:** What is the most fun neighborhood in Pittsburgh?

**My Sub-metric:** Number of active courts and rinks per neighborhood

**Dataset:** [City of Pittsburgh Courts and Rinks — WPRDC](https://data.wprdc.org/dataset/courts-and-rinks)

**Team Members:** Sachit Anand, Rhonda Ojongmboh, Osarumen Okhiku

---
## 1. Introduction

Courts and rinks represent the most sport-specific and competitive form of public recreation in Pittsburgh. Unlike parks or playgrounds, courts and rinks are built for structured athletic activity — basketball, tennis, hockey, volleyball, bocce, and more. Having more courts and rinks in a neighborhood means more opportunities for organized and pickup sports, which is one of the most direct expressions of community fun.

My contribution to the group's "most fun" metric is the count of active city-maintained courts and rinks per neighborhood. I specifically filter to active facilities only, since inactive courts cannot be used and should not count toward a fun score.

In this notebook I will:
1. Load and explore the City of Pittsburgh Courts and Rinks dataset
2. Filter to active courts and rinks only
3. Count per neighborhood and examine the types of courts available
4. Visualize the results
5. Identify the top neighborhood by courts and rinks count
6. Export a normalized score for use in the group combined metric

---
## 2. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

os.makedirs('data', exist_ok=True)
print('Libraries loaded successfully.')

---
## 3. Load the Data

In [ ]:
# Load from local CSV (place courts_rinks.csv inside a data/ folder next to this notebook)
courts = pd.read_csv('data/courts_rinks.csv')

print(f'Rows: {len(courts)}')
print(f'Columns: {list(courts.columns)}')
courts.head()

---
## 4. Explore the Data

In [ ]:
print(f'Total records in dataset: {len(courts)}')
print(f'\nInactive field value counts:')
print(courts['inactive'].value_counts())
print(f'\nMissing values per column:')
print(courts.isnull().sum())

In [ ]:
# What types of courts and rinks exist?
print('All court and rink types (full dataset):')
print(courts['type'].value_counts())

In [ ]:
# Filter to active courts only (inactive == 'f' means NOT inactive, i.e. active)
courts_active = courts[courts['inactive'] == 'f'].copy()

print(f'Active courts/rinks: {len(courts_active)}')
print(f'Inactive courts/rinks removed: {len(courts) - len(courts_active)}')
print(f'\nActive court types:')
print(courts_active['type'].value_counts())

In [ ]:
print(f'Unique neighborhoods with at least one active court/rink: {courts_active["neighborhood"].nunique()}')

---
## 5. Count Active Courts and Rinks per Neighborhood

In [ ]:
# Remove rows where neighborhood is missing
courts_clean = courts_active.dropna(subset=['neighborhood']).copy()
print(f'Rows after dropping missing neighborhoods: {len(courts_clean)}')

# Count active courts/rinks per neighborhood
courts_per_hood = (
    courts_clean
    .groupby('neighborhood')
    .size()
    .reset_index(name='courts_count')
    .sort_values('courts_count', ascending=False)
    .reset_index(drop=True)
)

print(f'\nNeighborhoods ranked by active courts/rinks count (top 15):')
print(courts_per_hood.head(15).to_string(index=False))

In [ ]:
# What types of courts does the top neighborhood have?
top_hood = courts_per_hood.iloc[0]['neighborhood']
print(f'Court/rink types in {top_hood}:')
print(
    courts_clean[courts_clean['neighborhood'] == top_hood]['type']
    .value_counts()
    .to_string()
)

---
## 6. Visualizations

In [ ]:
# ── Chart 1: Top 15 neighborhoods by courts/rinks count ──────────────────────
top15 = courts_per_hood.head(15)

fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#9b59b6' if i == 0 else '#3498db' for i in range(len(top15))]
bars = ax.bar(top15['neighborhood'], top15['courts_count'],
              color=colors, edgecolor='white', linewidth=0.8)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.set_title('Top 15 Pittsburgh Neighborhoods by Number of Active Courts and Rinks',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Neighborhood', fontsize=12)
ax.set_ylabel('Number of Active Courts / Rinks', fontsize=12)
ax.set_xticklabels(top15['neighborhood'], rotation=45, ha='right', fontsize=10)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#9b59b6', label='#1 Neighborhood'),
    plt.Rectangle((0,0),1,1, color='#3498db', label='Other Top 15')
], fontsize=10)

plt.tight_layout()
plt.savefig('courts_top15.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: courts_top15.png')

In [ ]:
# ── Chart 2: All neighborhoods ranked (horizontal) ───────────────────────────
sorted_all = courts_per_hood.sort_values('courts_count')

fig, ax = plt.subplots(figsize=(10, 18))
ax.barh(sorted_all['neighborhood'], sorted_all['courts_count'],
        color='#3498db', edgecolor='white')

ax.set_title('All Pittsburgh Neighborhoods Ranked by Active Courts/Rinks Count',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Number of Active Courts / Rinks', fontsize=12)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('courts_all_neighborhoods.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: courts_all_neighborhoods.png')

In [ ]:
# ── Chart 3: Stacked bar — court types for top 8 neighborhoods ───────────────
import numpy as np

top8_hoods = courts_per_hood.head(8)['neighborhood'].tolist()
top8_data  = courts_clean[courts_clean['neighborhood'].isin(top8_hoods)]

pivot = (
    top8_data.groupby(['neighborhood', 'type'])
    .size()
    .unstack(fill_value=0)
    .reindex(top8_hoods)
)

palette = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6',
           '#1abc9c','#e67e22','#34495e','#95a5a6','#d35400','#27ae60','#2980b9']

fig, ax = plt.subplots(figsize=(14, 7))
pivot.plot(kind='bar', stacked=True, ax=ax,
           color=palette[:len(pivot.columns)], edgecolor='white', linewidth=0.5)

ax.set_title('Court and Rink Types — Top 8 Neighborhoods',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Neighborhood', fontsize=12)
ax.set_ylabel('Number of Courts / Rinks', fontsize=12)
ax.set_xticklabels(top8_hoods, rotation=35, ha='right', fontsize=10)
ax.legend(title='Court Type', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('courts_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: courts_type_breakdown.png')

---
## 7. Results

In [ ]:
winner = courts_per_hood.iloc[0]
total_courts = courts_per_hood['courts_count'].sum()

print('=' * 56)
print(f'  Courts/Rinks Sub-metric Winner: {winner["neighborhood"]}')
print(f'  Active courts/rinks count: {winner["courts_count"]}')
print(f'  % of all active city courts: {winner["courts_count"]/total_courts*100:.1f}%')
print('=' * 56)
print()
print('Full top 5:')
print(courts_per_hood.head(5).to_string(index=False))

**Squirrel Hill South** dominates this sub-metric with 23 active courts and rinks — more than double the second-place Highland Park (20). It accounts for about 10% of all active city courts on its own. The type breakdown shows a diverse mix of basketball, tennis, and other court types, meaning there is something for a wide range of athletes.

---
## 8. Normalized Score for Group Combination

To combine fairly with Sachit's parks score and Rhonda's playgrounds score, the courts count is normalized to a 0–1 scale where 1.0 = the neighborhood with the most active courts (Squirrel Hill South, 23).

In [ ]:
courts_per_hood['courts_score'] = (
    courts_per_hood['courts_count'] / courts_per_hood['courts_count'].max()
)

# Export for use in the group notebook
courts_per_hood[['neighborhood', 'courts_count', 'courts_score']].to_csv(
    'data/courts_scores.csv', index=False
)

print('Exported: data/courts_scores.csv')
print()
print('Top 10 with normalized scores:')
print(courts_per_hood[['neighborhood', 'courts_count', 'courts_score']].head(10).to_string(index=False))

---
## 9. Conclusion

Based on the courts and rinks sub-metric alone, **Squirrel Hill South** is the most fun neighborhood in Pittsburgh with 23 active courts and rinks. This is a commanding lead over every other neighborhood in the city. Not only does it have the most facilities, it also has the most variety — basketball courts, tennis courts, hockey rinks, and more — which means it caters to the widest range of sports and recreational activities.

Of the three sub-metrics our group analyzed, courts and rinks arguably represent the most active and competitive form of fun. If a neighborhood scores highly here, it is a place where people regularly gather to play organized sports, which is a strong signal of community engagement and recreation.

**Personal reflection:** Squirrel Hill South as the top neighborhood for courts and rinks is both a data-driven result and one that I think reflects a real truth about the neighborhood. It is a dense, residential community with strong infrastructure investment. Personally, I might gravitate toward somewhere like the Strip District for nightlife-style fun, but when it comes to the kind of fun you can have every day for free with your neighbors — sports, pickup games, outdoor activity — Squirrel Hill South is clearly the most equipped neighborhood in the city, and that is a meaningful definition of fun.